In [2]:
import pandas as pd

def create_initial_df():

    df = pd.DataFrame(columns=['code'])
    print("创建了初始DataFrame，包含code列")
    return df
def add_column_to_df(df, column_name, code, default_value):
    
    df = df.copy()  # 避免修改原DataFrame
    
    # 检查列是否存在
    if column_name in df.columns:
        mask = df['code'] == code
        if mask.any():
            df.loc[mask, column_name] = default_value
        else:
            print(f"未找到code='{code}'的行")
    else:
        # 添加新列，默认值为空或NaN
        df[column_name] = pd.NA
        # 为特定code设置值
        mask = df['code'] == code
        if mask.any():
            df.loc[mask, column_name] = default_value
        else:
            print(f"未找到code='{code}'的行")
    
    return df

In [3]:
def calculate_interval(meet_time,leave_time):
    interval=leave_time-meet_time
    return interval
def calculate_barrier(meet_data,leave_data,df,code):
    num_barrier=len(meet_data[0])
    for i in range(num_barrier):
        #print('i:'+str(i))
        interval=0
        for j in range(16):
            #print('j:'+str(j))
            #print(meet_data[j][i])
            #print(leave_data[j][i])
            interval=calculate_interval(meet_data[j][i],leave_data[j][i])+interval;
        df=add_column_to_df(df, "barrier_"+str(i), code, interval//16);
    return df
            
    

In [4]:
import re
def extract_time_from_line(line):
    
    pattern = r'fthread\s+(\d+)\s+(start|end)\s+cal\s+at\s+(\d+)'
    match = re.search(pattern, line)
    if  match:
       return {
            'thread_id': int(match.group(1)),      # 线程号
            'action': match.group(2),              # 动作 (meet/leave)
            'time': int(match.group(3)),           # 时间
        }
    return None
def extract_dma_from_line(line):
    
    pattern = r'fthread\s+(\d+)\s+(start|end)\s+dma\s+(\d+)'
    
    match = re.search(pattern, line)
    if  match:
       return {
            'thread_id': int(match.group(1)),      # 线程号
            'action': match.group(2),              # 动作 (meet/leave)
            'time': int(match.group(3)),           # 时间
        }
    return None

def extract_time_from_txt_file(filename):

    flag=False
    time_value_start=0
    time_value_temp=0
    state="None"
    time_all=0
    dma_data = {}
    for i in range(16):
        dma_data[i] =0
    dma_all=0
    dma_time=0

    try:
        with open(filename, 'r', encoding='utf-8', errors='ignore') as file:
            for line_num, line in enumerate(file, 1):
                line = line.strip()
                res = extract_time_from_line(line)
                if res is not None:
                    if res['action']=='start':
                            if state!="start":
                                time_value_start=res['time']
                                time_all=time_value_temp+time_all
                                time_value_temp=0
                                state="start"

                    elif res['action']=='end':
                        time_value_temp=res['time']-time_value_start
                        if state!="end":
                            state="end"
                dma=extract_dma_from_line(line)
                if dma is not None:
                    if dma['action']=='start':
                        if(dma_data[dma['thread_id']]!=0):
                            print(line)
                            assert(0)
                        
                        dma_data[dma['thread_id']]= dma['time']
                    elif dma['action']=='end':
                        assert(dma_data[dma['thread_id']]!=0)
                        dma_all=dma_all+dma['time']-dma_data[dma['thread_id']]
                        dma_data[dma['thread_id']]=0
                        dma_time=dma_time+1
            time_all=time_value_temp+time_all
            for i in range(16):
                assert(dma_data[i]==0);
    except Exception as e:
        print(f"读取文件 {filename} 时出错: {e}")
      
    return  time_all, dma_all,dma_time


In [5]:
from pathlib import Path

def extract_thread_and_time(line):
    
    pattern = r'fthread\s+(\d+)\s+(meet|leave)\s+barrier(\d+)'
    pattern2 =  r'fthread\s+(\d+)\s+(start|end)\s+cal\s+at\s+(\d+)'
    match = re.search(pattern, line)
    if match:
        return {
            'thread_id': int(match.group(1)),      # 线程号
            'action': match.group(2),              # 动作 (meet/leave)
            'time': int(match.group(3)),           # 时间
        }
    else :
        match = re.search(pattern2, line)
        if match:
            return {
                'thread_id': int(match.group(1)),      # 线程号
                'action': match.group(2),              # 动作 (meet/leave)
                'time': int(match.group(3)),           # 时间
            }
        return None
def find_syn_file(filename):
    meet_data = {}
    for i in range(16):
        meet_data[i] = []
    leave_data = {}
    for i in range(16):
        leave_data[i] = []
    meet_data_aux = {}
    for i in range(16):
        meet_data_aux[i] = []
    leave_data_aux = {}
    for i in range(16):
        leave_data_aux[i] = []

    count=0
    state="None"
    
    try:
        with open(filename, 'r', encoding='utf-8', errors='ignore') as file:
            for line_num, line in enumerate(file, 1):
                line = line.strip()
                res=extract_thread_and_time(line)
                if res is not None:
                    if res['action']=='meet':
                        meet_data[res['thread_id']].append(res['time'])
                    elif res['action']=='leave':
                        leave_data[res['thread_id']].append(res['time'])
                    elif res['action']=='start':
                        leave_data_aux[res['thread_id']].append(res['time'])
                        if state!="start":
                            for i in range(16):
                                meet_data_aux[i].append(res['time'])
                            state="start"
                        count=count+1;
                    elif res['action']=='end':
                        meet_data_aux[res['thread_id']].append(res['time'])
                        if state!="end":
                            assert(count==16)
                            state="end"
                        count=count-1
                        if count==0:
                            for i in range(16):
                                leave_data_aux[i].append(res['time'])
                    else:
                        assert(0)
        assert(state=="end")
        for i in range(16):
            meet_data[i]=meet_data[i]+meet_data_aux[i]
            leave_data[i]=leave_data[i]+leave_data_aux[i]
    except Exception as e:
        print(f"读取文件 {filename} 时出错: {e}")
    
    return meet_data,leave_data


In [6]:
import os
import glob
from pathlib import Path
import csv
import pandas as pd
import re
def main(path_):
    """使用os.listdir遍历当前目录中包含'txt'的文件"""
    df=create_initial_df()
    for filename in os.listdir(path_):
        if 'txt' in filename:
            # 按下划线分割，取第一部分
            name_part = filename.split('_')[0]
            new_row = pd.DataFrame({'code': [str(name_part)]})
            df = pd.concat([df, new_row], ignore_index=True)
            time_value,dma_value,dma_time = extract_time_from_txt_file(path_+filename)
            df=add_column_to_df(df, "total_cycle", str(name_part), time_value);
            df=add_column_to_df(df, "dma_cost", str(name_part), (dma_value//16));
            df=add_column_to_df(df, "dma_times", str(name_part), dma_time);
            meet_data,leave_data=find_syn_file(path_+filename)
            
            df= calculate_barrier(meet_data,leave_data,df,str(name_part)); 
            print(filename+" is finsihed")     
    df.to_csv(path_+'result.csv', index=False, encoding='utf-8')
main("vmmu/")
main("mmu/")

创建了初始DataFrame，包含code列
SEL_output.txt is finsihed
BFS_output.txt is finsihed
RED_output.txt is finsihed
GEMV_output.txt is finsihed
BS_output.txt is finsihed
VA_output.txt is finsihed
HST-S_output.txt is finsihed
TS_output.txt is finsihed
MLP_output.txt is finsihed
NW_output.txt is finsihed
创建了初始DataFrame，包含code列
SEL_output.txt is finsihed
BFS_output.txt is finsihed
RED_output.txt is finsihed
GEMV_output.txt is finsihed
BS_output.txt is finsihed
HST-S_output.txt is finsihed
TS_output.txt is finsihed
VA.txt is finsihed
MLP_output.txt is finsihed
NW_output.txt is finsihed
